<a href="https://colab.research.google.com/github/LeslieF03/Calculadora-de-procesos-markovianos-de-decisi-n/blob/main/Procesos_Markovianos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1.   El primer bloque corresponde a las funciones usadas en el programa
2.   El segundo  a la funcion del menu

In [ ]:
!pip install pulp

In [ ]:
import numpy as np
import pulp as p

from pulp import LpProblem, LpMinimize, LpMaximize, LpVariable, lpSum, LpStatus, value

def ev(entrada):
    #Funcion que solo transforma fracciones flotantes, o bien no devuelve nada si el numero ingresado es valido.... - _ -
    try:
        if isinstance(entrada, str):
            resultado = eval(entrada)
            return resultado
        elif isinstance(entrada, (int, float)):
            return entrada
        else:
            return "Entrada no válida. Por favor, introduce una cadena o un número."
    except Exception as e:
        return f"Error al evaluar la operación: {e}"

class Datos:
    #Esta clase se encarga de la lectura de datos, NOTA, algunos datos se crean como matrices que contienen matrices,  lo que puede ser MUUY dificil la lectura
    #se debe tener cuidado En este caso:::::
    #       M_Costo=[num_Edos x num_Decicsiones] una sola matriz de costos
    #       K= [num_Decisiones][ num_Edos X num_Edos]  una lista de matrices de decision
    #       R= [num_Politicas][num_Edos] Lista de politicas, cada politica tiene una lista de decisiones en cada estado
    #       Matrices_Politica = [num_Politicas] [num_Edos X num_Edos] lista que para cada politica genera una matriz de transicion
    #       Vector_Proba_Estacionaria, Para el metodo de busqueda exahustiva...
    #       Objetivo
    #


    def __init__(self):
        #Datos relacionados al tamaño de matrices y listas
        self.num_Decisiones=int(0)
        self.num_Edos=int(0)
        self.num_Politicas=int(0)
        #Matrices de decisiones, politicas, estados, vectorEstacionario
        self.K=[] #lista donde se guardan desiciones
        self.R=[] #lista donde se guardan políticas
        self.M_Costo=np.empty((0, 0)) # Matriz de costos, se inicializa como vacia
        self.Matrices_Politica=[] #Se guardaran las matrices por cada politica aqui.
        self.Vector_Proba_Estacionaria=[]
        self.Objetivo=0 #1 si es Maximizar, 0 si es Minimizar, por defecto incia en 0, se modifica en def dimensiones
        self.variables_Y = {} ###########Para guardar los nombres de las variables para el PPL

    def dimensiones(self):
        print("  >>ESCRIBA SUS DATOS")
        print("")
    # se pregunta por el número de estados, políticas y estados viables.
        self.num_Edos = int(input("¿Cuántos Estados tienes?            "))
        self.num_Decisiones = int(input("¿Cuántas Decisiones tienes?         "))
        self.num_Politicas = int(input("¿Cuántas políticas viables tienes?  "))
        self.Objetivo = int(input("Maximizar=1 o Minimizar=0:          "))

        return [self.num_Edos,self.num_Decisiones,self.num_Politicas,self.Objetivo,self.variables_Y]

    def Creador_Matrices_Decision(self):  # Aprobado
    # Rellenar las matrices de transición de acuerdo a las decisiones válidas
        self.Aplica = np.full((self.num_Edos, self.num_Decisiones), "N", dtype='<U1')
        for l in range(1, self.num_Decisiones + 1):
            print(" ")
            print("- - - - - - - - - - - - - - - -")
            print(f"        MATRIZ DE DECISIÓN K={l}...")
            print("- - - - - - - - - - - - - - - -")
            Matrix = np.zeros((self.num_Edos, self.num_Edos))  # Crear una matriz de ceros (tamaño num_Edos x num_Edos)

            for i in range(self.num_Edos):
                aplicaPolitica = str(input(f">>>¿La decisión {l} es viable en el estado {i}?(Y/N):    ")).upper()
                self.Aplica[i,l-1]=aplicaPolitica

                var_name = f"y{i}{l}"  # Crear variable Yil (por ejemplo, y03 si estado=0, decisión=3)

                if aplicaPolitica == "Y":

                    self.variables_Y[(i, l)] = p.LpVariable(var_name, lowBound=0, cat=p.LpContinuous)

                    print(f"     --->Proba del estado {i} ...")
                    for j in range(self.num_Edos):
                        val = False
                        while not val:
                            probabilidad = ev(input(f"                              al  Edo. {j}:  "))
                            if 0 <= probabilidad <= 1:
                                Matrix[i, j] = probabilidad
                                val = True
                            else:
                                print("----Solo Valores entre 0 y 1----")
                else:
                    Matrix[i, :] = 0  # Ceros en las políticas inviables
                    var = p.LpVariable(var_name, lowBound=0, upBound=0, cat=p.LpContinuous)  # ← Ya queda forzada a cero
                    self.variables_Y[(i, l)] = var

            self.K.append(Matrix)  # Guardar la matriz de transición para la política l

        print("-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-")
        return self.K, self.Aplica

    def Creador_politicas(self): #Aprobado
        print("DEFINICIÓN DE POLÍTICAS:")
        for l in range(0,self.num_Politicas):
            print(" - - - - - - - - - - - - -")
            print(f"Para la política {l+1}:")
            politica=[]
            politica.pop
            for i in range (0, self.num_Edos):
                politica.append(int(input(f"decisión tomada en Edo.{i}:   ")))
            self.R.append(politica)
        return self.R

    def Creador_Matrices_Costos(self): #Aprobado
        print("* * * * * * * * * * * * * * * *")
        print("MATRIZ DE COSTOS:")
        print("* * * * * * * * * * * * * * * *")
        Matrix=np.zeros((self.num_Edos,self.num_Decisiones))
        for i in range (0,self.num_Edos):
            for j in range (0,self.num_Decisiones):
                    Matrix[i,j]=float(input(f"Costo de estar en {i} y tomar la decisión {j+1}:  "))
            print(" - - - - - - - - - - - - -")
        self.M_Costo=Matrix
        return self.M_Costo

    def Creador_Matrices_Por_Politica(self):
        for l in range(0,self.num_Politicas):
            politica=self.R[l]

            Matrix=np.zeros((self.num_Edos,self.num_Edos))
            for i in range (0,self.num_Edos):
                Desicion=self.K[politica[i]-1]
                Matrix[i,:]=Desicion[i,:]
            self.Matrices_Politica.append(Matrix)
        return self.Matrices_Politica

class Busqueda_Exahustiva:

    def __init__(self,num_decisiones,num_edos,num_politicas,Objetivo,K_,R_,Costo,Matrices_Politica):
        #Datos relacionados al tamaño de matrices y listas
        self.num_Decisiones=int(num_decisiones)
        self.num_Edos=int(num_edos)
        self.num_Politicas=int(num_politicas)
        #Matrices de decisiones, politicas, estados, vectorEstacionario
        self.K=K_ #lista donde se guardan desiciones
        self.R=R_ #lista donde se guardan políticas
        self.M_Costo=Costo # Matriz de costos, se inicaliza como vacia
        self.Matrices_Politica=Matrices_Politica #Se guardaran las matrices por cada politica aqui.
        self.Vector_Proba_Estacionaria=[]
        self.Objetivo=Objetivo #True  si es Maximizar, False si es Minimizar

    def Calcular_Vector_Proba_Estacionaria(self):
        #Como lo indica su nombre, solo calcula el vector de proba estacionaria

        for l in range(0,self.num_Politicas):
            Matrix=self.Matrices_Politica[l]
            Sistema_Pi= np.transpose(Matrix) #Transpone la matriz de transicion original
            for i in range(0,self.num_Edos):
                Sistema_Pi[i,i]=Sistema_Pi[i,i]-1 # Luego en la diagonal le restamos 1, pues se simula que pasa restando el Pi correspondiente
            Sistema_Pi[self.num_Edos - 1, :] = 1 # y esto es por la condicion de que todas las pi's suman uno  :)
            b = np.zeros(self.num_Edos)
            b[-1] = 1  # aqui el -1, nos lleva al ultimo elemento, como toda pi's suma 1, le asiganmos dicho 1
            self.Vector_Proba_Estacionaria.append(np.linalg.solve(Sistema_Pi, b))

        print("- - - - - - - - - - - - - - - -")

    def Salida2(self):
        costosFinales=[0]*self.num_Politicas
        for l in range(0,self.num_Politicas):
            politica=self.R[l]
            #print("politica::")
            #print(politica)
            VectorEstacionario=self.Vector_Proba_Estacionaria[l]


            for i in range (0, self.num_Edos):
                costosFinales[l]+=self.M_Costo[i,politica[i]-1]*VectorEstacionario[i]

            print(f" -> Vector PI para la  política {politica} es: {VectorEstacionario} ")
            print(f"--> El costo esperado en la política{politica} es : {costosFinales[l]}")
            print(" ")
        if self.Objetivo==1:
            i=costosFinales.index(max( costosFinales))
            print("- - - - - - - - - - - - - - - -")
            print(f"#####La política Máxima es: {self.R[i]} con un coste de {costosFinales[i]} #####")
            print("- - - - - - - - - - - - - - - -")
        elif self.Objetivo==0:
            i=costosFinales.index(min( costosFinales))
            print("- - - - - - - - - - - - - - - -")
            print(f"#####La política Minima es: {self.R[i]} con un coste de {costosFinales[i]} #####")
            print("- - - - - - - - - - - - - - - -")

    def Salida1(self):
        print(" ")
        print("*- - - - - - -#SOLUCIÓN# - - - - - - *")
        for l in range(0,self.num_Politicas):
            politica=self.R[l]
            print("- - - - - - - - - - - - - - - -")
            print(f"MATRIZ DE LA POLÍTICA R{l+1}={politica}...")
            print(self.Matrices_Politica[l])
        print("- - - - - - - - - - - - - - - -")

class MejoraPoliticas:
    def __init__(self,num_decisiones,num_edos,num_politicas,Objetivo,K_,R_,Costo,Matrices_Politica):
        #Datos relacionados al tamaño de matrices y listas
        self.num_Decisiones=int(num_decisiones)
        self.num_Edos=int(num_edos)
        self.num_Politicas=int(num_politicas)
        #Matrices de decisiones, politicas, estados, vectorEstacionario
        self.K=K_ #lista donde se guardan desiciones
        self.R=R_ #lista donde se guardan políticas
        self.M_Costo=Costo # Matriz de costos, se inicaliza como vacia
        self.Matrices_Politica=Matrices_Politica #Se guardaran las matrices por cada politica aqui.
        self.Objetivo=Objetivo #True  si es Maximizar, False si es Minimizar
        self.Politica=[]
        self.PoliticaAnterior=[]
        self.Sistema_Vi_gR=np.zeros((self.num_Edos, self.num_Edos)) # Matriz que nos calculara el sistema
        self.Sistema_Vi_gR[:, self.num_Edos-1]=[1]*self.num_Edos
        self.Cons_Vi_gR=[0]*self.num_Edos
        self.solsist=[]
        self.sol=0


    def definirPoliticaInical(self):
        print("Sus politicas son:")
        for i,polit in enumerate(self.R):
            print(f"  *Politica {i+1}={polit}")
        i=int(input("Escriba el número correspondiente a su politica inicial:"))
        self.Politica=self.R[i-1]
        print("- - - - - - - - - - - - - - - - ")
        print(f"Politica inicial: {self.Politica}")
        print("- - - - - - - - - - - - - - - - ")
    def DeterminarValores(self):
        for i in range(0, self.num_Edos):
            self.Cons_Vi_gR[i]= self.M_Costo[i, self.Politica[i]-1]
            Matrix= self.K[self.Politica[i]-1]

            for j in range (0, self.num_Edos-1):
                self.Sistema_Vi_gR[i,j]=-1*Matrix[i,j]
                if j==i:
                    self.Sistema_Vi_gR[i,j]+=1
        print("- - - - - - - - - - - - - - - - ")
        print("Sistema Generado:")
        print(self.Sistema_Vi_gR)
        print(" = ")
        print(f"{[float(elemento) for elemento in self.Cons_Vi_gR]}T")
        print("- - - - - - - - - - - - - - - - ")
        self.solsist=(np.linalg.solve(self.Sistema_Vi_gR, self.Cons_Vi_gR).tolist())
        print("- - - - - - - - SOLUCIÓN SISTEMA - - - - - - - -")
        print("")
        for i in range(0,len(self.solsist)-1):
            print(f"  *V{i}= {self.solsist[i]}")
        print(f"  *g(R)= {self.solsist[-1]}")
        self.sol=self.solsist[-1]
        self.solsist[-1]=0

    def Mejorar(self):
        newPolitica=[0]*self.num_Edos
        for l in range (0,self.num_Edos):
            Valores=[]
            indices=[]
            print("- - - - - - - - - - -")
            for k in range (0,self.num_Decisiones):
                if  not np.all(self.K[k][l, :] == 0):
                    val=self.M_Costo[l,k]-self.solsist[l]
                    for i in range (0, self.num_Edos):
                        val+=self.solsist[i]*self.K[k][l,i]
                    print(f"     i={l}; k={k+1} ------> V{l}={val}")
                    Valores.append(val)
                    indices.append(k)
            if self.Objetivo==0:
                newPolitica[l]=indices[Valores.index(min(Valores))]+1
            else:
                newPolitica[l]=indices[Valores.index(max(Valores))]+1
            print("- - - - - - - - - - -")

        if newPolitica!=self.Politica:
            self.Politica=newPolitica
            print(":::::::::::::::::::::::::::::::::::::::::::::")
            print(f">>La nueva politica Optima es:  {newPolitica}")
            print(":::::::::::::::::::::::::::::::::::::::::::::")
            return True
        elif newPolitica==self.Politica:
            print(":::::::::::::::::::::::::::::::::::::::::::::")
            print(f">>La Solución es la politica Optima:  {newPolitica}")
            print(f"Con un coste de {self.sol}")
            print(":::::::::::::::::::::::::::::::::::::::::::::")
            return False

class PPL_proce:

    def __init__(self,num_decisiones,num_edos,num_politicas,Objetivo,K_,R_,Costo,Matrices_Politica, variables_Y):
        #Datos relacionados al tamaño de matrices y listas
        self.num_Decisiones=int(num_decisiones)
        self.num_Edos=int(num_edos)
        self.num_Politicas=int(num_politicas)
        #Matrices de decisiones, politicas, estados, vectorEstacionario
        self.K=K_ #lista donde se guardan decisiones
        self.R=R_ #lista donde se guardan políticas
        self.M_Costo=Costo # Matriz de costos, se inicializa como vacia
        self.Matrices_Politica=Matrices_Politica #Se guardaran las matrices por cada politica aqui.
        self.Vector_Proba_Estacionaria=[]
        self.Objetivo=Objetivo #True  si es Maximizar, False si es Minimizar
        self.variables_Y = variables_Y

        # Crear el modelo
        if self.Objetivo==0:
            modelo = LpProblem("MiModelo", LpMinimize)
        else:
            modelo = LpProblem("MiModelo", LpMaximize)

        modelo += lpSum(
            self.M_Costo[i][l-1] * var
            for (i, l), var in self.variables_Y.items()
        ), "Funcion Objetivo"

        # Restricción 1: la suma total de todas las variables debe ser igual a 1
        modelo += lpSum(var for var in self.variables_Y.values()) == 1, "R1"
        #print(modelo)

        for j in range(self.num_Edos):  # Una restricción por estado j
            modelo += (
                lpSum(self.variables_Y[(j, l)] for l in range(1, self.num_Decisiones + 1))
                - lpSum(
                    self.variables_Y[(i, l)] * self.K[l - 1][i][j]
                    for l in range(1, self.num_Decisiones + 1)
                    for i in range(self.num_Edos)
                )
                == 0,
                f"R{j+2}"
            )

        # Guardar el modelo en el objeto
        self.modelo = modelo

        print("- - - - - - - - - - - - - - - -")
        print(modelo)
        print("- - - - - - - - - - - - - - - -")

        # Resolver el modelo
        modelo.solve()

        # Mostrar estado de la solución
        print("Estado de la solución:", LpStatus[modelo.status])
        print("")

        # Mostrar valores óptimos de las variables Yil
        print("Valores óptimos de las variables Yil:")
        for (i, l), var in self.variables_Y.items():
            print(f"y{i}{l} = {var.varValue}")

        Poli_min = []
        for (i, l), var in self.variables_Y.items():
            if var.varValue > 0:
                Poli_min.append(l)

        print("- - - - - - - - - - - - - - - -")
        print(f"#####La política optima es: {Poli_min} con un coste de {p.value(modelo.objective)} #####")
        print("- - - - - - - - - - - - - - - -")

class MejoraPoliticasDescuento:
    def __init__(self,num_decisiones,num_edos,num_politicas,Objetivo,K_,R_,Costo,Matrices_Politica):
        #Datos relacionados al tamaño de matrices y listas
        self.num_Decisiones=int(num_decisiones)
        self.num_Edos=int(num_edos)
        self.num_Politicas=int(num_politicas)
        self.K=K_ #lista donde se guardan desiciones
        self.R=R_ #lista donde se guardan políticas
        self.M_Costo=Costo # Matriz de costos, se inicaliza como vacia
        self.Matrices_Politica=Matrices_Politica #Se guardaran las matrices por cada politica aqui.
        self.Objetivo=Objetivo #True  si es Maximizar, False si es Minimizar
        self.Politica=[]
        self.PoliticaAnterior=[]
        self.FactorDeDescuento=0
        self.Sistema_Vi=np.zeros((self.num_Edos, self.num_Edos)) # Matriz que nos calculara el sistema
        self.Cons_Vi=[0]*self.num_Edos

    def definirPoliticaInical(self):
        print("- - - - - - - - - - - - - - - - ")
        self.FactorDeDescuento=np.float64(input("Escriba su Factor de descuento Alpha:    "))
        print("- - - - - - - - - - - - - - - - ")
        print("Sus politicas son:")
        for i,polit in enumerate(self.R):
            print(f"  *Politica {i+1}={polit}")
        i=int(input("Escriba el número correspondiente a su politica inicial:"))
        self.Politica=self.R[i-1]
        print("- - - - - - - - - - - - - - - - ")
        print(f"Politica inicial: {self.Politica}")
        print("- - - - - - - - - - - - - - - - ")
    def DeterminarValores(self):
        for i in range(0, self.num_Edos):
            self.Cons_Vi[i]=self.M_Costo[i, self.Politica[i]-1]
            Matrix= self.K[self.Politica[i]-1]
            for j in range (0, self.num_Edos):
                self.Sistema_Vi[i,j]=-1*self.FactorDeDescuento*Matrix[i,j]
                if j==i:
                    self.Sistema_Vi[i,j]+=1
        print("- - - - - - - - - - - - - - - - ")
        print("Sistema Generado:")
        print(self.Sistema_Vi)
        print(" = ")
        print(f"{[float(elemento) for elemento in self.Cons_Vi]}T")
        print("- - - - - - - - - - - - - - - - ")
        self.solsist=(np.linalg.solve(self.Sistema_Vi, self.Cons_Vi).tolist())
        print("- - - - - - - - SOLUCIÓN SISTEMA - - - - - - - -")
        print("")
        for i in range(0,len(self.solsist)):
            print(f"  *V{i}= {self.solsist[i]}")

    def Mejorar(self):
        newPolitica=[0]*self.num_Edos
        for l in range (0,self.num_Edos):
            Valores=[]
            indices=[]
            print("- - - - - - - - - - -")
            for k in range (0,self.num_Decisiones):
                if  not np.all(self.K[k][l, :] == 0):
                    val=self.M_Costo[l,k]
                    for i in range (0, len(self.solsist)):
                        val+=self.FactorDeDescuento*self.solsist[i]*self.K[k][l,i]
                    print(f"     i={l}; k={k+1} ------> V{l}={val}")
                    Valores.append(val)
                    indices.append(k)
            if self.Objetivo==0:
                newPolitica[l]=indices[Valores.index(min(Valores))]+1
            else:
                newPolitica[l]=indices[Valores.index(max(Valores))]+1
            print("- - - - - - - - - - -")

        if newPolitica!=self.Politica:
            self.Politica=newPolitica
            print(":::::::::::::::::::::::::::::::::::::::::::::")
            print(f">>La nueva politica Optima es:  {newPolitica}")
            print(":::::::::::::::::::::::::::::::::::::::::::::")
            return True
        elif newPolitica==self.Politica:
            print(":::::::::::::::::::::::::::::::::::::::::::::")
            print(f">>La Solución es la politica Optima:  {newPolitica}")
            print(":::::::::::::::::::::::::::::::::::::::::::::")
            return False

class AproximacionesSucesivas:
    def __init__(self,num_decisiones,num_edos,num_politicas,Objetivo,K_,R_,Costo,Matrices_Politica, Aplica):
        #Datos relacionados al tamaño de matrices y listas
        self.num_Decisiones=int(num_decisiones)
        self.num_Edos=int(num_edos)
        self.num_Politicas=int(num_politicas)
        self.K=K_ #lista donde se guardan desiciones
        self.R=R_ #lista donde se guardan políticas
        self.M_Costo=Costo # Matriz de costos
        self.Matrices_Politica=Matrices_Politica #Se guardaran las matrices por cada politica aqui.
        self.Objetivo=Objetivo #True  si es Maximizar, False si es Minimizar
        self.Politica=[]
        self.Aplica=Aplica
        self.Descuento=0
        self.max_it=0
        self.tolerancia=0
        self.V0=[]
        self.Vn = [0] * self.num_Edos
        self.Vn_anterior = self.Vn
        self.sol=[0] * self.num_Edos
        self.politica_Iteracion=[0]* self.num_Edos

    def condiciones_paro(self):
            print("- - - - - - - - - - - - - - - - ")
            self.Descuento=np.float64(input("Escriba su Factor de descuento Alpha:    "))
            print("- - - - - - - - - - - - - - - - ")
            self.max_it = int(input("Escriba el número máximo de iteraciones:    "))
            self.tolerancia=np.float64(input("Escriba la tolerancia:    "))
            print("- - - - - - - - - - - - - - - - ")

    def cumple_tolerancia(self, Vn, Vn_anterior) -> bool:
         print("######################################################")
         print("Tolerancia=", self.tolerancia)
         print("")
         Vn2= [float(x) for x in Vn]
         Vn_anterior2= [float(x) for x in Vn_anterior]
         print("    Vn=", Vn2)
         print("    Vn_anterior=", Vn_anterior2)
         print("")
         resta= [abs(v - a) for v, a in zip(Vn, Vn_anterior)]
         print("Resta =", [float(x) for x in resta])
         if all(abs(v - a) <= self.tolerancia for v, a in zip(Vn, Vn_anterior)):
             print("########## Se cumplio con la tolerancia ##########")
             return True
         else:
             print("###################################################")
             return False

    def primeraIteracion(self):
        print("-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_")
        print("               n=1")
        print("-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_")
        self.politica_Iteracion=[0]*self.num_Edos
        self.V0=[0]*self.num_Edos
        for i in range(self.num_Edos):
            fila_costos = self.M_Costo[i, :]
            fila_aplica = self.Aplica[i, :]
            costos_validos = fila_costos[fila_aplica == "Y"]

            if self.Objetivo==1:
                self.V0[i] = max(costos_validos)
                print(f"                       -->V_{i}^1 = Max {costos_validos}")
            else:
                self.V0[i] = min(costos_validos)
                print(f"                       -->V_{i}^1 = Min{costos_validos}")
            indices_coincidencia = np.where(fila_costos == self.V0[i])[0]
            coincidencia = indices_coincidencia[0] if indices_coincidencia.size > 0 else None
            self.politica_Iteracion[i]=coincidencia+1

            print(f"                       -->V_{i}^1 = {self.V0[i]}")
            print(" ")
        print("###################################################")
        politica1= [int(x) for x in self.politica_Iteracion]
        print(f"            -->Política de la iteración 1: {politica1}")
        print("###################################################")
        self.Vn_anterior=self.V0
        self.max_it -= 1


    def Iteracion(self):
        nuevo_valor = 0
        m=2

        while self.max_it > 0:
            print("-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_")
            print(f"               n={m}")
            print("-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_")
            self.politica_Iteracion=[0]*self.num_Edos
            for i in range(self.num_Edos):
                valores_viables = []
                deciciones=[]
                for k in range(1, self.num_Decisiones + 1):
                    if self.Aplica[i, k-1] == "Y":
                        suma = 0
                        for j in range(self.num_Edos):
                            #print(f"C_{i}{k} = ", float(self.M_Costo[i, k-1]))
                            #print(f"p_{i}{j}({k}) = ", float(self.K[k-1][i, j]))
                            #print(f"V_{j}^n-1 = ",float( self.Vn_anterior[j]))
                            suma += self.K[k-1][i, j] * self.Vn_anterior[j]
                        nuevo_valor = self.M_Costo[i, k-1] + self.Descuento * suma
                        #print(f"                {k}) Suma = ", nuevo_valor)
                        valores_viables.append(nuevo_valor)
                        deciciones.append(k)
                        valores_viables2 = [float(x) for x in valores_viables]

                print("")
                if valores_viables:
                    if self.Objetivo==1:
                        print(f"            -->V{i}^{m} = Max {valores_viables2}")
                        self.Vn[i] = max(valores_viables)
                        self.politica_Iteracion[i]= deciciones[valores_viables.index(max(valores_viables))]
                    else:
                        print(f"            -->V{i}^{m} = Min {valores_viables2}")
                        self.Vn[i] = min(valores_viables)
                        self.politica_Iteracion[i]= deciciones[valores_viables.index(min(valores_viables))]
                    print(f"            -->V_{i}^{m} = {self.Vn[i]}")
                    print("")
            print("")
            print("###################################################")
            print(f"Politica de la iteración {m}: {self.politica_Iteracion}")
            if self.cumple_tolerancia(self.Vn, self.Vn_anterior):
                self.max_it = 0
                break
            self.Vn_anterior = self.Vn[:]
            self.max_it -= 1
            m+=1
        print("")
        print(":::::::::::::::::::::::::::::::::::::::::::::::::::")
        print(f"  Politica Final: {self.politica_Iteracion}")
        print(":::::::::::::::::::::::::::::::::::::::::::::::::::")

'''
    def Iteracion(self):
        nuevo_valor = 0
        print("\n n = 2\n")

        while self.max_it > 0:
            for i in range(self.num_Edos):
                valores_viables = []
                indices_viables = []
                for k in range(1, self.num_Decisiones + 1):
                    if self.Aplica[i, k-1] == "Y":
                        suma = 0
                        for j in range(self.num_Edos):
                            print(f"C_{i}{k} = ", self.M_Costo[i, k-1])
                            print(f"p_{i}{j}({k}) = ", self.K[k-1][i, j])
                            print(f"V_{j}^n-1 = ", self.Vn_anterior[j])
                            suma += self.K[k-1][i, j] * self.Vn_anterior[j]
                        nuevo_valor = self.M_Costo[i, k-1] + self.Descuento * suma
                        #print("suma = ", nuevo_valor)
                        valores_viables.append(nuevo_valor)
                        indices_viables.append(k)
                        print("politica: ",indices_viables)
                        print(f"min{valores_viables}")
                if valores_viables:
                    if self.Objetivo==True:
                        self.Vn[i] = max(valores_viables)
                    else:
                        self.Vn[i] = min(valores_viables)
                    print(f"V_{i}^{self.max_it + 1} = {self.Vn[i]}")
            print("Vn = ", self.Vn)
            print("Vn anterior = ", self.Vn_anterior)
            if self.cumple_tolerancia(self.Vn, self.Vn_anterior):
                self.max_it = 0
                break
            self.Vn_anterior = self.Vn[:]
            self.max_it -= 1
            print("- - - - - - - - - - - - - - - -")
            print(f"n = ")
'''
#Falta agregar tolerancia y solución

'\n    def Iteracion(self):\n        nuevo_valor = 0\n        print("\n n = 2\n")\n\n        while self.max_it > 0:\n            for i in range(self.num_Edos):\n                valores_viables = []\n                indices_viables = []\n                for k in range(1, self.num_Decisiones + 1):\n                    if self.Aplica[i, k-1] == "Y":\n                        suma = 0\n                        for j in range(self.num_Edos):\n                            print(f"C_{i}{k} = ", self.M_Costo[i, k-1])\n                            print(f"p_{i}{j}({k}) = ", self.K[k-1][i, j])\n                            print(f"V_{j}^n-1 = ", self.Vn_anterior[j])\n                            suma += self.K[k-1][i, j] * self.Vn_anterior[j]\n                        nuevo_valor = self.M_Costo[i, k-1] + self.Descuento * suma\n                        #print("suma = ", nuevo_valor)\n                        valores_viables.append(nuevo_valor)\n                        indices_viables.append(k)\n          

In [ ]:
#funciones del programa
def BusquedaEx(numDecisiones,numEdos, NumPoliticas, Objetivo, MatricesDecision, Politicas, Costos, MatricesPolitica):
    Busqueda_Ex=Busqueda_Exahustiva(numDecisiones,numEdos, NumPoliticas, Objetivo, MatricesDecision, Politicas, Costos, MatricesPolitica)
    Busqueda_Ex.Salida1()
    Busqueda_Ex.Calcular_Vector_Proba_Estacionaria()
    Busqueda_Ex.Salida2()

def MejoraPol(numDecisiones,numEdos, NumPoliticas, Objetivo, MatricesDecision, Politicas, Costos, MatricesPolitica):
    MejoraPol=MejoraPoliticas(numDecisiones,numEdos, NumPoliticas, Objetivo, MatricesDecision, Politicas, Costos, MatricesPolitica)
    MejoraPol.definirPoliticaInical()
    condicion=True
    while condicion==True:
        print("")
        MejoraPol.DeterminarValores()
        condicion=MejoraPol.Mejorar()

def MejoraPolDesc(numDecisiones,numEdos, NumPoliticas, Objetivo, MatricesDecision, Politicas, Costos, MatricesPolitica):
    MejoraPolD=MejoraPoliticasDescuento(numDecisiones,numEdos, NumPoliticas, Objetivo, MatricesDecision, Politicas, Costos, MatricesPolitica)
    MejoraPolD.definirPoliticaInical()
    condicion=True
    while condicion==True:
        print("")
        MejoraPolD.DeterminarValores()
        condicion=MejoraPolD.Mejorar()

def PPL(numDecisiones,numEdos, NumPoliticas, Objetivo, MatricesDecision, Politicas, Costos, MatricesPolitica, variables_Y):
    Prog_Lineal=PPL_proce(numDecisiones,numEdos, NumPoliticas, Objetivo, MatricesDecision, Politicas, Costos, MatricesPolitica, variables_Y)

def AproxSucesivas(numDecisiones,numEdos, NumPoliticas, Objetivo, MatricesDecision, Politicas, Costos, MatricesPolitica, Aplica):
    Aproximaciones=AproximacionesSucesivas(numDecisiones,numEdos, NumPoliticas, Objetivo, MatricesDecision, Politicas, Costos, MatricesPolitica, Aplica)
    Aproximaciones.condiciones_paro()
    Aproximaciones.primeraIteracion()
    Aproximaciones.Iteracion()

# **MENU DEL PROGRAMA:**

Se recomienda poner solo las funciones y trozos de codigo desarrollados en su respectivo caso

In [1]:
def menu():
    #INICIO DE MENU:
    print("")
    print("          <<CALCULADORA DE PROCESOS MARKOVIANOS DE DECISIÓN>>")
    print("")
    Dat=False
    while True:
        print("\n--- Menú Principal ---")
        print("0. Meter Datos")
        print("1. Búsqueda exhaustiva")
        print("2. Solución por PPL")
        print("3. Mejoramiento de políticas")
        print("4. Mejoramiento de políticas con descuento")
        print("5. Aproximaciones sucesivas")
        print("6. Salir")

        try:
            print("- - - - - - - - - - - - - -")
            opcion = int(input("Elige una opción (0-6): "))
            if opcion == 0 and Dat==False:
              ### SOLICITUD DE DATOS ###
              print("  ")
              Data=Datos()
              numEdos,numDecisiones,NumPoliticas, Objetivo, variables_Y = Data.dimensiones()
              print("  ")
              print(f"Decisiones:{numDecisiones}, Edos:{numEdos}, Polticas:{NumPoliticas}, objetivo:{Objetivo}")
              MatricesDecision,Aplica=Data.Creador_Matrices_Decision()
              for i,matrix in enumerate(MatricesDecision):
                  print(f"Matriz de decisión K={i+1}")
                  print(matrix)
              Costos=Data.Creador_Matrices_Costos()
              print(Costos)
              Politicas=Data.Creador_politicas()
              MatricesPolitica=Data.Creador_Matrices_Por_Politica()
              print("    ")
              Dat=True
            if opcion == 1 and Dat==True:
                print(">>>Has elegido: Búsqueda exhaustiva")
                BusquedaEx(numDecisiones,numEdos, NumPoliticas, Objetivo, MatricesDecision, Politicas, Costos, MatricesPolitica)
                input(" ")
            elif opcion == 2 and Dat==True:
                print(">>>Has elegido: Solución por PPL")
                PPL(numDecisiones,numEdos, NumPoliticas, Objetivo, MatricesDecision, Politicas, Costos, MatricesPolitica, variables_Y) #########################
            elif opcion == 3 and Dat==True:
                print(">>>Has elegido: Mejoramiento de políticas")
                MejoraPol(numDecisiones,numEdos, NumPoliticas, Objetivo, MatricesDecision, Politicas, Costos, MatricesPolitica)
            elif opcion == 4 and Dat==True:
                print(">>>Has elegido: Mejoramiento de políticas Con Descuento")
                MejoraPolDesc(numDecisiones,numEdos, NumPoliticas, Objetivo, MatricesDecision, Politicas, Costos, MatricesPolitica)
            elif opcion == 5 and Dat==True:
                print(">>>Has elegido: Aproximaciones sucesivas")
                AproxSucesivas(numDecisiones,numEdos, NumPoliticas, Objetivo, MatricesDecision, Politicas, Costos, MatricesPolitica, Aplica)
            elif opcion == 6:
                print(">>>Saliendo del programa. ¡Adiós!")
                break
            else:
                if Dat==False:
                  print("No tiene datos para Iniciar los algoritmos")
        except ValueError:
            print("Entrada no válida. Por favor, ingresa un número.")
#Menu Del programa

menu()


          <<CALCULADORA DE PROCESOS MARKOVIANOS DE DECISIÓN>>


--- Menú Principal ---
0. Meter Datos
1. Búsqueda exhaustiva
2. Solución por PPL
3. Mejoramiento de políticas
4. Mejoramiento de políticas con descuento
5. Aproximaciones sucesivas
6. Salir
- - - - - - - - - - - - - -
Elige una opción (0-6): 6
>>>Saliendo del programa. ¡Adiós!
